## CV-Summarizer

Tell me your story and get a clever on-point headline for your CV

In [ ]:
import json
import openai
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display

load_dotenv(override=True)

API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = "gpt-5-nano"
WEBSITE_URL = "https://edwarddonner.com/"

In [ ]:
def stream_chat(messages):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )
    display_handle = display(Markdown(""), display_id=True)
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        update_display(Markdown(result), display_id=display_handle.display_id)
    return result

## Version A: From Story

In [ ]:
def summarize_cv(message):
    return stream_chat([
        {"role": "system", "content": """
            You a snarky and humorous career counselor and CV helper.
            People usually come to you with their story of life or content of the personal website.
            Your task is to help them to write a one-line phrase for the heading of their CV.
            Just write the headline without any comments.
        """},
        {"role": "user", "content": message}
    ])

In [ ]:
mock_story = stream_chat([
    {"role": "user", "content": """
        Write the story of life from a freshly graduated student.
        You can choose study of field and invent previous milestones in life.
        Tell the story just as the person might tell it after a few beers in a bar.
        Just write the story in one paragraph without any comments.
    """}
])

In [ ]:
headline = summarize_cv(mock_story)

## Version B: From Website

In [ ]:
from scraper import fetch_website_links, fetch_website_contents

LINK_SELECTOR_SYSTEM_PROMPT = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

def get_links_user_prompt(url):
    user_prompt = f"""
        Here is the list of links on the website {url} -
        Please decide which of these are relevant web links for a describing the person behind the website.
        Respond with the full https URL in JSON format.
        Do not include Terms of Service, Privacy, Contact Information

        Links (some might be relative links):

    """
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": LINK_SELECTOR_SYSTEM_PROMPT},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result or "{}")
    return links

def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link["type"]}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
prompt = f"""
    Hi, here is the content of my website and other relevant pages:
    {fetch_page_and_all_relevant_links(WEBSITE_URL)}

    Use this information to write me a snappy headline for my CV.
"""[:5000]
headline = summarize_cv(prompt)